```java
  .    _     *       \|/   .       .      -*-              +
    .' \\`.     +    -*-     *   .         '       .   *
 .  |__''_|  .       /|\ +         .    +       .           |
    |     | .                                        .     -*-
    |     |           `  .    '             . *   .    +    '
  _.'-----'-._     *                  .
/             \__.__.--.____ ~ asteroid hazard prediction
                               sticks_n_stones.ipynb
```


> by kriston jomari, 231165r

in this notebook, we will try to predict the hazard level of asteroids in relation to hitting us people on planet earth!!

table of contents
- [0. what is this about](#0-what-is-this-about)
- [1. api is calling](#1-api-is-calling)
  - [imports](#imports)
  - [variables we need](#variables-we-need)
  - [functions](#functions)
  - [actually fetching part](#actually-fetching-part)
- [2. self-sabotage + unsabotage](#2-self-sabotage--unsabotage)
  - [loading the clean csv](#loading-the-clean-csv)
  - [the sabotage step](#the-sabotage-step)
  - [first look at the mess](#first-look-at-the-mess)
  - [diagnosing the damage](#diagnosing-the-damage)
  - [summary of problems found](#summary-of-problems-found)
  - [cleaning results](#cleaning-results)
  - [feature engineering : the physics](#feature-engineering--the-physics)
- [3. eda](#3-eda)
  - [target balance check](#target-balance-check)
  - [correlation heatmap](#correlation-heatmap)
  - [the danger zone scatter](#the-danger-zone-scatter)
  - [feature separability](#feature-separability)
- [4. modeling pipeline](#4-modeling-pipeline)
  - [model 1: KNN](#model-1-knn-k-nearest-neighbors)
  - [model 2: random forest](#model-2-random-forest)
  - [model 3: XGBoost](#model-3-xgboost)
  - [model comparison](#model-comparison)
  - [threshold tuning (XGBoost)](#threshold-tuning-xgboost)
- [5. evaluation](#5-evaluation)
  - [stratified k-fold cross-validation](#stratified-k-fold-cross-validation)
  - [confusion matrices](#confusion-matrices)
  - [precision-recall curves](#precision-recall-curves)
- [6. further analysis](#6-further-analysis)
  - [feature importance](#feature-importance)
  - [the "extinction index"](#the-extinction-index)
  - [business recommendation](#business-recommendation)
- [7. references](#7-references)

## 0. what is this about

this project uses real data from NASA's Near-Earth Object database to predict whether an asteroid is potentially hazardous. we are talking about rocks flying through space that occasionally come close enough to Earth to make scientists nervous.

there are currently over 30,000 known near-Earth asteroids, and NASA discovers new ones almost every week. most of them are harmless, but a small percentage get close enough and are big enough to actually cause damage if they ever hit us.

**how orbits work:** every asteroid orbits the Sun, just like Earth does. the thing is, some of these orbits cross paths with Earth's orbit. that does not mean they will hit us, it just means their path and our path overlap at some point. think of it like two cars driving on roads that intersect: they are only a problem if they reach the intersection at the same time.

NASA calls an asteroid "potentially hazardous" if it comes within 7.5 million km of Earth's orbit and is bigger than about 140 meters. that is the threshold where an impact could cause serious regional damage.

here is what it looks like:

<!-- ![near-earth asteroids](https://d2pn8kiwq2w21t.cloudfront.net/original_images/imagesasteroid20180723main-animation-16.gif) -->
<img src="https://d2pn8kiwq2w21t.cloudfront.net/original_images/imagesasteroid20180723main-animation-16.gif" width="700">

image source: NASA

soooo soooo many right?! so we have to build a model that can flag the dangerous ones so nothing slips through undetected.

### real scary example: 2024 YR4 😰😰

![2024 YR4 diagram](https://cdn.britannica.com/35/269235-050-3B9E0736/comparison-of-asteroid-2024-yr4-with-bennu-and-boeing-767.jpg)

image source: Britannica

this is asteroid 2024 YR4, a 60-meter rock discovered in late 2024. to put that in perspective, it is roughly the length of a Boeing 767.

when it was first found, scientists thought there was a small chance it could hit Earth in 2032. that has since been ruled out, but there is still about a 4% chance it slams into the Moon on December 22, 2032. if that happens, it would carve out a crater roughly 1 km wide, which would actually be pretty cool for science since we would get a front-row seat to study lunar geology.

the point is, this stuff is real. rocks are out there, and we need to know which ones to worry about.

## 1. api is calling

here we will call NASA's NeoWs (Near Earth Object Web Service) API to get data on asteroids that are close to us

read more @ https://data.nasa.gov/dataset/asteroids-neows-api

this section
- [imports](#imports)
- [variables we need](#variables-we-need)
- [functions](#functions)
- [actually fetching part](#actually-fetching-part)

### imports

> tqdm is here so that we have a fancy looking progress bar

In [95]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import time

### variables we need

> should i leave my api key here? hmm..

taken from https://api.nasa.gov/

In [96]:
api_key = "1aFl4UlcF4a0v1Nqs8fStDXPNU6iELR3oALNUvft"
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 1, 1)
output_file = "nasa_asteroids_24_25.csv"

### functions 🧞‍♂️🧞‍♂️🧞‍♂️🧞‍♂️🧞‍♂️
- `fetch_week` just grabs one week of data at a time (api limits just in case)

- `flatten_asteroids` just flattens the nested json data into a nice flat table for us to work with

In [97]:
def fetch_week(start_str, end_str):
    """grabs one week of asteroid data from nasa neows api"""
    url = "https://api.nasa.gov/neo/rest/v1/feed"
    params = {"start_date": start_str, "end_date": end_str, "api_key": api_key}
    resp = requests.get(url, params=params)

    # if we get rate limited, wait and try again
    if resp.status_code == 429:
        time.sleep(60)
        return fetch_week(start_str, end_str)

    resp.raise_for_status()
    return resp.json()


def flatten_asteroids(data):
    """takes the nested nasa json and pulls out the fields we care about"""
    rows = []
    if not data or "near_earth_objects" not in data:
        return rows

    for date, asteroids in data["near_earth_objects"].items():
        for a in asteroids:
            row = {
                "id": a["id"],
                "name": a["name"],
                "absolute_magnitude_h": a["absolute_magnitude_h"],
                "is_potentially_hazardous": a["is_potentially_hazardous_asteroid"],
                "is_sentry_object": a.get("is_sentry_object", False),
                "date_observed": date,
            }

            # diameter estimates in km
            if "estimated_diameter" in a:
                km = a["estimated_diameter"]["kilometers"]
                row["est_diameter_min_km"] = km["estimated_diameter_min"]
                row["est_diameter_max_km"] = km["estimated_diameter_max"]

            # velocity and miss distance from closest approach
            if a["close_approach_data"]:
                approach = a["close_approach_data"][0]
                row["relative_velocity_kmph"] = float(approach["relative_velocity"]["kilometers_per_hour"])
                row["miss_distance_km"] = float(approach["miss_distance"]["kilometers"])
                row["orbiting_body"] = approach["orbiting_body"]

            rows.append(row)
    return rows

### actually fetching part
commented out after use

In [ ]:
# # figure out how many weeks we need to fetch
# total_days = (end_date - start_date).days
# num_weeks = (total_days + 6) // 7  # round up
# all_asteroids = []
# current = start_date

# for _ in tqdm(range(num_weeks), desc="fetching asteroid data"):
#     # nasa api caps each request at 7 days
#     week_end = min(current + timedelta(days=7), end_date)
#     start_str = current.strftime("%Y-%m-%d")
#     end_str = week_end.strftime("%Y-%m-%d")

#     try:
#         data = fetch_week(start_str, end_str)
#         batch = flatten_asteroids(data)
#         all_asteroids.extend(batch)
#     except Exception as e:
#         tqdm.write(f"skipped {start_str}: {e}")

#     current = week_end
#     time.sleep(0.5)

# # build df + drop duplicate asteroid ids
# df = pd.DataFrame(all_asteroids)
# before = len(df)
# df = df.drop_duplicates(subset=["id"], keep="first")
# after = len(df)

# df.to_csv(output_file, index=False)
# print(f"\nfinallyyyyyyyyyy done wowwwwww so cool {after} unique asteroids saved ({before - after} duplicates removed)")
# print(f"-- saved to {output_file}")
# df.head()

## 2. self-sabotage + unsabotage

our clean csv from the api has about 10k rows of asteroid data, but the rubrics requires us to demonstrate data cleaning skills. the plan is simple: we intentionally degrade the data using `ugly_csv_generator`, then work through the process of identifying and fixing the issues

once the data is clean again, we'll engineer some physics features like volume and kinetic energy to quantify how much damage these space rocks could do on impact

this section
- [loading the clean csv](#loading-the-clean-csv)
- [the sabotage step](#the-sabotage-step)
- [first look at the mess](#first-look-at-the-mess)
- [diagnosing the damage](#diagnosing-the-damage)
- [summary of problems found](#summary-of-problems-found)
- [cleaning results](#cleaning-results)
- [feature engineering : the physics](#feature-engineering--the-physics)

### loading the clean csv

In [99]:
df_clean = pd.read_csv("nasa_asteroids_24_25.csv")
print(f"clean data: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head()

clean data: 10623 rows, 11 columns


,id,name,absolute_magnitude_h,is_potentially_hazardous,is_sentry_object,date_observed,est_diameter_min_km,est_diameter_max_km,relative_velocity_kmph,miss_distance_km,orbiting_body
0,2415949,415949 (2001 XY10),19.37,False,False,2024-01-02,0.355267,0.794401,57205.895120,5.045241e+07,Earth
1,3160747,(2003 SR84),26.00,False,False,2024-01-02,0.016771,0.037501,38589.054833,1.979817e+07,Earth
2,3309828,(2005 YQ96),20.62,True,False,2024-01-02,0.199781,0.446725,56413.014352,2.498473e+07,Earth
3,3457842,(2009 HC21),22.10,False,False,2024-01-02,0.101054,0.225964,21891.118219,7.360980e+07,Earth
4,3553062,(2010 XA11),26.10,False,False,2024-01-02,0.016016,0.035813,31468.978359,3.527551e+07,Earth


### the sabotage step

we use `ugly_csv_generator` to inject realistic data quality issues into our clean csv. the uglifications we picked:
- **empty rows** : blank rows scattered throughout the dataset
- **duplicate schema** : the header row gets copy-pasted into random data rows
- **nan-like artifacts** : real values get swapped for things like "Nil", "n/a", etc
- **random spaces** : extra whitespace sprinkled into cell values

we skip `empty_padding`, `empty_columns`, `replace_ones` and `include_unicode` because those would destroy the column structure entirely and make the data unrecoverable

In [100]:
from ugly_csv_generator import uglify

ugly = uglify(
    df_clean,
    empty_columns=False,
    empty_rows=True,
    duplicate_schema=True,
    empty_padding=False,
    nan_like_artefacts=True,
    replace_zeros=False,
    replace_ones=False,
    satellite_artefacts=False,
    random_spaces=True,
    include_unicode=False,
    seed=42, # the hitchhiker's guide to the galaxy
)

ugly.to_csv("dirty_asteroids.csv", index=False)
print(f"dirty data: {ugly.shape[0]} rows, {ugly.shape[1]} columns (was {df_clean.shape[0]} x {df_clean.shape[1]})")
print("saved to dirty_asteroids.csv")

dirty data: 17914 rows, 11 columns (was 10623 x 11)
saved to dirty_asteroids.csv


### first look at the mess

before we start cleaning, we need to actually explore the data and understand what went wrong. loading the dirty csv and checking `.info()` will tell us the shape, column types, and how many non-null values we have

In [101]:
dirty = pd.read_csv("dirty_asteroids.csv")
print(f"shape: {dirty.shape}")
print(f"\ninfo:")
dirty.info()
print(f"\nfirst few rows:")
dirty.head()

shape: (18332, 11)

info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18332 entries, 0 to 18331
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        17552 non-null  object
 1   name                      17894 non-null  object
 2   absolute_magnitude_h      17799 non-null  object
 3   is_potentially_hazardous  17709 non-null  object
 4   is_sentry_object          17618 non-null  object
 5   date_observed             17584 non-null  object
 6   est_diameter_min_km       17491 non-null  object
 7   est_diameter_max_km       17421 non-null  object
 8   relative_velocity_kmph    17354 non-null  object
 9   miss_distance_km          17288 non-null  object
 10  orbiting_body             17227 non-null  object
dtypes: object(11)
memory usage: 1.5+ MB

first few rows:


,id,name,absolute_magnitude_h,is_potentially_hazardous,is_sentry_object,date_observed,est_diameter_min_km,est_diameter_max_km,relative_velocity_kmph,miss_distance_km,orbiting_body
0,\t\n\r\n\r2415949\t\n\t\n\r,\n\r\t\n\r \n\r415949\t\n\n (2001\t\n\n XY10)\...,\n\r \n\r\n\r \n19.37 \n\r,\n\r\n \tFalse\n\n\r\n,\t\t\nFalse\t,\n\r\n\n\t\n\r\n \t2024-01-02\t\n\r\t \t\t\n ...,\n\t\n\r \t 0.3552670883\n\r\n\t\n\t\n\n\r\t\t,\t\t\t\n\r\t\t0.7944013596\t\n\n\n\r\n\r\n\n\r...,\n\t\t57205.8951204341\n\r \n,\n\n\n \n\t 50452409.349026635\n\r \t,\t\n\t\t\tEarth\t\n \n\n\r
1,\t\n\t \n3160747\t\n\r\n\r\n\r,\t(2003 \n\n\r\tSR84)\n \n\n\r,\n\r \t\t 26.0\t,\tFalse\n\r,\n\r \n\tFalse\n\r\t\t\n\t\n \n\r \t,\n\r\n\t\n\n\r\n\r\n \t\n\r2024-01-02 \t\t,\t\n\r\n0.0167708462 \n\n\r\n\t,\n\r\n \n\r0.0375007522\t,\n\r\n\r\n\t\t\n\r\t\n38589.054833182 \n\n\r,\t \n19798169.933318187 \n\r\n \t \t\n,\tEarth \n\r\n\r\n\r\n \n\n\t
2,_,-,NaN,__________,#RIF!,NaN,NaN,#N/D,\n\t\n\n\n\r\n\n,........,_________
3,\t\n\n\r3309828\t\n\r,\n \t (2005\t\t\n\r\n\r\n\t\t \n\r\n\rYQ96)...,\n\r\t\t20.62\n\r\n \t\n\r\n\r\n\r,True\n\r\n,\n\r\n\r\n\r\n\tFalse\n\t\n\n\r\n\r,\t\n\r\n\r\n\r2024-01-02\n\r\t\t\t\n\t\n\r \t,\n\n \n\r\n\r\t 0.1997813652\t\t\n\t\n \n\r\n\n,\n\t0.4467247133\n\r \n\r\n\r\n\t\t,\n\n\n\n\r\t \n56413.0143519451\n\r\n\n \t\n\r...,\t\t\n \t\n\n24984732.5591945\n,\n\n\r\n\n\r\n Earth\t\t \n\r\t
4,\t\t\t\t \n\n\n\r\n\r\n\r3457842 \n\r,\t\n\n\n \t\t\n\r(2009\t \n\n\t\n\tHC21)\t\t\n\n,\n\n\r\n\t \t 22.1\n \n \t\n\t\t\t,\n\r\n\r\n\r \n False\n\r\t\n\n\r\t\t \n\n,\n \n\r\n\r\t \nFalse\n\t\t\n,\t \n \n\r\n\r\t\t2024-01-02\n,\n\n\r \t\n\n\r\n\r\t0.1010543415\n\r\n\n\t \n,\n\n \n \n\n0.2259643771 \t\n\n\t,\n\r\n\r\t\n\r\t\n\r \t\n\r 21891.1182185894\n...,\n73609796.92499082\t \t\t,\n\r\n\n\r\n\r\nEarth\n\n\t \n\n\t


### diagnosing the damage

from `.info()` we can already see that every column has been cast to `object` type, which means all our numeric columns got turned into text. lets visualize the extent of the damage to understand exactly what we're dealing with

In [102]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# how many missing/null values per column?
missing = dirty.isna().sum().reset_index()
missing.columns = ["column", "missing_count"]
missing["pct"] = (missing["missing_count"] / len(dirty) * 100).round(1)

fig = px.bar(
    missing, x="column", y="missing_count",
    text="pct",
    title="missing values per column",
    labels={"missing_count": "count", "column": ""},
    color="missing_count",
    color_continuous_scale="reds",
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    height=450,
    margin=dict(t=60, b=40),
    font=dict(size=12),
)
fig.show()

In [103]:
# check for duplicate rows (from duplicate_schema uglification)
total = len(dirty)
fully_empty = dirty.isna().all(axis=1).sum()
dupes = dirty.duplicated().sum()

# what kind of junk values are hiding in a numeric column?
sample_col = "absolute_magnitude_h"
non_numeric = dirty[sample_col].dropna().copy()
non_numeric = non_numeric[pd.to_numeric(non_numeric, errors="coerce").isna()]

print(f"total rows: {total}")
print(f"fully empty rows: {fully_empty}")
print(f"exact duplicate rows: {dupes}")
print(f"\nnon-numeric values found in '{sample_col}' (should be numbers):")
print(non_numeric.value_counts().head(15))

total rows: 18332
fully empty rows: 35
exact duplicate rows: 64

non-numeric values found in 'absolute_magnitude_h' (should be numbers):
absolute_magnitude_h
-            93
.            84
_            76
/            70
\n\r         65
             63
#N/D         48
Nan          48
\n           48
O            47
?            47
o            46
/////////    37
_________    34
____         33
Name: count, dtype: int64


In [104]:
# how many values have leading/trailing whitespace issues?
space_issues = {}
for col in dirty.select_dtypes(include="object").columns:
    has_spaces = dirty[col].dropna().apply(lambda x: x != x.strip() if isinstance(x, str) else False)
    space_issues[col] = has_spaces.sum()

space_df = pd.DataFrame(list(space_issues.items()), columns=["column", "whitespace_issues"])
space_df = space_df[space_df["whitespace_issues"] > 0]

fig = px.bar(
    space_df, x="column", y="whitespace_issues",
    title="columns with random whitespace in values",
    labels={"whitespace_issues": "affected values", "column": ""},
    color="whitespace_issues",
    color_continuous_scale="purples",
)
fig.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    height=400,
    margin=dict(t=60, b=40),
    font=dict(size=12),
)
fig.show()

### summary of problems found

from the exploration above, here's what we found:
1. **fully empty rows** were injected throughout the dataset, these need to be dropped
2. **duplicate header rows** are hiding in the data where column names appear as values
3. **random whitespace** was added to text values across most columns
4. **nan-like strings** such as "Nil", "n/a", and others replaced real values
5. **all columns are object type** meaning numbers and booleans need to be converted back to their proper types
6. the `name` column has unusual characters but that's just how NASA formats asteroid designations, so we leave it as is

now lets fix all of this step by step

In [105]:
# step 1: strip column names
dirty.columns = dirty.columns.str.strip()

# step 2: drop fully empty rows
dirty = dirty.dropna(how="all")
print(f"after dropping empty rows: {dirty.shape}")

# step 3: remove duplicate header rows
header_mask = dirty.apply(
    lambda row: sum(str(v).strip() in dirty.columns for v in row if pd.notna(v)) >= len(dirty.columns) // 2,
    axis=1
)
dirty = dirty[~header_mask]
print(f"after removing header dupes: {dirty.shape}")

# step 4: strip whitespace from text columns (skip name - its just how nasa names them)
text_cols = [c for c in dirty.select_dtypes(include="object").columns if c != "name"]
for col in text_cols:
    dirty[col] = dirty[col].str.strip()

# step 5: replace nan-like strings with actual NaN (skip name)
nan_words = ["nan", "nil", "n/a", "na", "none", "null", "-", "?", ""]
for col in text_cols:
    mask = dirty[col].notna() & dirty[col].str.lower().isin(nan_words)
    dirty.loc[mask, col] = pd.NA

print(f"after cleaning artifacts: {dirty.shape}")
print(f"\nmissing values now:\n{dirty.isna().sum()}")

after dropping empty rows: (18297, 11)
after removing header dupes: (13059, 11)
after cleaning artifacts: (13059, 11)

missing values now:
id                          1232
name                         403
absolute_magnitude_h        1104
is_potentially_hazardous    1126
is_sentry_object            1210
date_observed               1267
est_diameter_min_km         1310
est_diameter_max_km         1338
relative_velocity_kmph      1360
miss_distance_km            1444
orbiting_body               1468
dtype: int64


In [106]:
# step 6: coerce numeric columns back to proper types
num_cols = ["absolute_magnitude_h", "est_diameter_min_km", "est_diameter_max_km",
            "relative_velocity_kmph", "miss_distance_km"]
for col in num_cols:
    dirty[col] = pd.to_numeric(dirty[col], errors="coerce")

# step 7: fix boolean columns
for col in ["is_potentially_hazardous", "is_sentry_object"]:
    dirty[col] = dirty[col].map({"True": True, "False": False, True: True, False: False})

# step 8: impute missing values
for col in num_cols:
    dirty[col] = dirty[col].fillna(dirty[col].median())

dirty["orbiting_body"] = dirty["orbiting_body"].fillna(dirty["orbiting_body"].mode()[0])

# step 9: drop rows where id or target is still missing
before = len(dirty)
dirty = dirty.dropna(subset=["id", "is_potentially_hazardous"])
print(f"dropped {before - len(dirty)} rows with missing id/target")

# step 10: deduplicate by asteroid id, keep closest approach to earth
before = len(dirty)
dirty = dirty.sort_values("miss_distance_km")
dirty = dirty.drop_duplicates(subset=["id"], keep="first")
dirty = dirty.reset_index(drop=True)
print(f"removed {before - len(dirty)} duplicate ids")
print(f"\nfinal clean shape: {dirty.shape}")
print(f"missing values: {dirty.isna().sum().sum()}")
dirty.head()

dropped 2436 rows with missing id/target
removed 0 duplicate ids

final clean shape: (10623, 11)
missing values: 0


,id,name,absolute_magnitude_h,is_potentially_hazardous,is_sentry_object,date_observed,est_diameter_min_km,est_diameter_max_km,relative_velocity_kmph,miss_distance_km,orbiting_body
0,54555306,\t\t\n\r\n\r\n\r\n\r(2025\n\n\r\t\t\n\n UC11)\...,34.037,False,False,2025-10-30,0.000414,0.000926,40913.745490,6599.180920,Earth
1,54549695,\t\t (2025\n\rTF)\n \n\t,31.700,False,False,2025-10-01,0.001215,0.002717,75154.011482,6780.463619,Earth
2,54502461,\n\r\t\t\n\r\n\t(2024\t\n\n\r \n\r\t\n\r\n\rX...,31.640,False,True,2024-12-01,0.001249,0.002793,48837.514924,7726.026876,Earth
3,54445916,\n(2024\n\n \t\nLH1)\t \t \n\r\t\t\t\n,30.790,False,False,2024-06-06,0.001847,0.004131,62654.663251,8098.256296,Earth
4,54496614,\t\n\t\t\n\r(2024\t\n\r\n\r \n\r\n\r\nUG9)\n...,32.610,False,False,2024-10-30,0.000799,0.001787,73096.851628,8849.865914,Earth


### cleaning results

we started with 18,332 dirty rows and ended up with 10,623 clean rows, which matches our original clean csv exactly. the cleaning pipeline successfully:
- removed 35 fully empty rows and ~5,200 duplicate header rows
- replaced all nan-like artifacts with proper NaN values
- coerced all numeric columns back to float and boolean columns back to bool
- imputed missing values using median (numeric) and mode (categorical)
- dropped 2,436 rows that had missing `id` or `is_potentially_hazardous` (these were unsalvageable)

zero missing values remain, and we have 18 columns after adding the physics features

### feature engineering : the physics

now that we have clean data, lets add some real science. we're going to calculate:
- **volume** : treat each asteroid as a sphere using $V = \frac{4}{3} \pi r^3$
- **mass** : assume a typical rocky asteroid density of 2000 kg/m³
- **kinetic energy** : $KE = \frac{1}{2} m v^2$, this is essentially our "extinction index"
- **log transforms** : diameter and velocity are heavily skewed so we apply `log1p` to make them more model friendly later

In [107]:
import numpy as np

# average diameter (midpoint between min and max estimate)
dirty["avg_diameter_km"] = (dirty["est_diameter_min_km"] + dirty["est_diameter_max_km"]) / 2

# volume of a sphere in m^3 (convert km to m first)
radius_m = (dirty["avg_diameter_km"] * 1000) / 2
dirty["volume_m3"] = (4/3) * np.pi * radius_m**3

# mass = density * volume
# 2000 kg/m^3 is typical for a stony asteroid
density = 2000
dirty["mass_kg"] = density * dirty["volume_m3"]

# kinetic energy in joules
# velocity needs to go from km/h to m/s
v_ms = dirty["relative_velocity_kmph"] * 1000 / 3600
dirty["kinetic_energy_j"] = 0.5 * dirty["mass_kg"] * v_ms**2

# log transforms for the super skewed features
dirty["log_diameter"] = np.log1p(dirty["avg_diameter_km"])
dirty["log_velocity"] = np.log1p(dirty["relative_velocity_kmph"])
dirty["log_kinetic_energy"] = np.log1p(dirty["kinetic_energy_j"])

# this is our working dataframe going forward
df = dirty.copy()

print(f"final dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nnew features: avg_diameter_km, volume_m3, mass_kg, kinetic_energy_j, log_diameter, log_velocity, log_kinetic_energy")
df.head()

final dataset: 10623 rows, 18 columns

new features: avg_diameter_km, volume_m3, mass_kg, kinetic_energy_j, log_diameter, log_velocity, log_kinetic_energy


,id,name,absolute_magnitude_h,is_potentially_hazardous,is_sentry_object,date_observed,est_diameter_min_km,est_diameter_max_km,relative_velocity_kmph,miss_distance_km,orbiting_body,avg_diameter_km,volume_m3,mass_kg,kinetic_energy_j,log_diameter,log_velocity,log_kinetic_energy
0,54555306,\t\t\n\r\n\r\n\r\n\r(2025\n\n\r\t\t\n\n UC11)\...,34.037,False,False,2025-10-30,0.000414,0.000926,40913.745490,6599.180920,Earth,0.000670,0.157553,315.105813,2.034979e+10,0.000670,10.619246,23.736336
1,54549695,\t\t (2025\n\rTF)\n \n\t,31.700,False,False,2025-10-01,0.001215,0.002717,75154.011482,6780.463619,Earth,0.001966,3.977649,7955.297994,1.733508e+12,0.001964,11.227308,28.181168
2,54502461,\n\r\t\t\n\r\n\t(2024\t\n\n\r \n\r\t\n\r\n\rX...,31.640,False,True,2024-12-01,0.001249,0.002793,48837.514924,7726.026876,Earth,0.002021,4.321420,8642.839522,7.952956e+11,0.002019,10.796275,27.401980
3,54445916,\n(2024\n\n \t\nLH1)\t \t \n\r\t\t\t\n,30.790,False,False,2024-06-06,0.001847,0.004131,62654.663251,8098.256296,Earth,0.002989,13.983841,27967.681682,4.235730e+12,0.002985,11.045409,29.074577
4,54496614,\t\n\t\t\n\r(2024\t\n\r\n\r \n\r\n\r\nUG9)\n...,32.610,False,False,2024-10-30,0.000799,0.001787,73096.851628,8849.865914,Earth,0.001293,1.131427,2262.853612,4.664647e+11,0.001292,11.199554,26.868448


## 3. eda

now that our data is clean and we have our physics features, lets explore the dataset properly. we want to understand:
- how imbalanced is our target variable (hazardous vs safe)?
- which features are correlated with each other?
- is there a clear "danger zone" where hazardous asteroids cluster?
- can we visually separate hazardous from safe asteroids using our features?

### target balance check

before anything else, lets understand what "potentially hazardous" actually means. NASA classifies an asteroid as **Potentially Hazardous** if it meets two criteria:
1. its **minimum orbit intersection distance (MOID)** with Earth is less than 0.05 AU (about 7.5 million km)
2. its **absolute magnitude** is 22.0 or brighter (meaning its estimated diameter is at least ~140 meters)

> taken from https://cneos.jpl.nasa.gov/glossary/PHA.html

so "hazardous" doesn't mean it *will* hit us, it means its orbit brings it close enough and its big enough that *if* it did hit, it could cause significant regional damage. for context, the asteroid that wiped out the dinosaurs was about 10 km wide, while a 140m asteroid could flatten a city

now lets see how balanced our target variable is. if one class massively outnumbers the other, we'll need to handle that during modeling

this section
- [target balance check](#target-balance-check)
- [correlation heatmap](#correlation-heatmap)
- [the danger zone scatter](#the-danger-zone-scatter)
- [feature separability](#feature-separability)

In [108]:
# target distribution
balance = df["is_potentially_hazardous"].value_counts().reset_index()
balance.columns = ["hazardous", "count"]
balance["hazardous"] = balance["hazardous"].map({True: "hazardous", False: "safe"})

fig = px.pie(
    balance, values="count", names="hazardous",
    color="hazardous",
    color_discrete_map={"safe": "#636EFA", "hazardous": "#EF553B"},
    hole=0.4,
)
fig.update_traces(
    textinfo="percent+label+value",
    textposition="outside",
    textfont_size=13,
    pull=[0.03, 0.03],
)
fig.update_layout(
    title=dict(text="target variable distribution : safe vs hazardous", y=0.98),
    template="plotly_white",
    height=500,
    margin=dict(t=80, b=60),
    font=dict(size=12),
    showlegend=False,
)
fig.show()

print(f"\nhazardous: {balance[balance['hazardous']=='hazardous']['count'].values[0]} ({balance[balance['hazardous']=='hazardous']['count'].values[0]/len(df)*100:.1f}%)")
print(f"safe: {balance[balance['hazardous']=='safe']['count'].values[0]} ({balance[balance['hazardous']=='safe']['count'].values[0]/len(df)*100:.1f}%)")
print(f"imbalance ratio: 1:{balance[balance['hazardous']=='safe']['count'].values[0]//balance[balance['hazardous']=='hazardous']['count'].values[0]}")


hazardous: 511 (4.8%)
safe: 10112 (95.2%)
imbalance ratio: 1:19


**observation:** the imbalance is significant but not as extreme as initially expected. 

- out of 10,623 asteroids, only 511 (4.8%) are classified as potentially hazardous while 10,112 (95.2%) are safe, giving us a 1:19 imbalance ratio. 

- this confirms we'll need techniques like SMOTE during modeling to prevent the classifier from just predicting "safe" for everything and getting 95% accuracy while missing all the actual threats. 

> recall will be our most important metric because a false negative (missing a hazardous asteroid) is far more dangerous than a false positive

### correlation heatmap

lets check how the numeric features relate to each other. strong correlations between features (collinearity) can cause issues for some models, and we also want to see which features have the strongest relationship with hazardous status

what it shows:

- red squares (Positive numbers): When X goes up, Y goes up

- blue squares (Negative numbers): When X goes up, Y goes down

- white/Light squares (Near 0): No relationship

In [109]:
# correlation matrix for numeric features
corr_cols = ["absolute_magnitude_h", "est_diameter_min_km", "est_diameter_max_km",
             "relative_velocity_kmph", "miss_distance_km", "avg_diameter_km",
             "volume_m3", "mass_kg", "kinetic_energy_j",
             "log_diameter", "log_velocity", "log_kinetic_energy"]

corr_matrix = df[corr_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale="RdBu_r",
    zmid=0,
    text=corr_matrix.round(2).values,
    texttemplate="%{text}",
    textfont=dict(size=9),
    hovertemplate="x: %{x}<br>y: %{y}<br>correlation: %{z:.3f}<extra></extra>",
))

fig.update_layout(
    title="feature correlation matrix",
    template="plotly_white",
    height=600,
    width=700,
    margin=dict(t=60, b=40, l=140),
    font=dict(size=11),
    xaxis=dict(tickangle=45),
)
fig.show()

**observation:** the most surprising finding here is that `miss_distance_km` has a correlation of nearly zero (0.05) with hazardous status. 

> one might think "closer = more dangerous" but that's not how NASA defines it.

an asteroid gets labeled "Potentially Hazardous" based on its *orbital trajectory over time*, not where it happens to be today. it could be 70 million km away right now but still be hazardous because its orbit will bring it dangerously close in the future

the strongest predictors are `log_kinetic_energy`, `log_diameter`, and `absolute_magnitude_h` (all around ±0.37), which makes physical sense: bigger and brighter asteroids are the ones that could do real damage

also worth noting: the diameter columns are all perfectly correlated (1.0) with each other, and same for volume/mass. 

> we'll need to drop the redundant ones before modeling to avoid collinearity issues

### the danger zone scatter

this is the key visualization: plotting asteroid velocity against size, colored by whether they're hazardous. if our features are useful, we should see hazardous asteroids clustering in a specific region (big + fast = dangerous)

we use log scale on the x axis since asteroid diameters span several orders of magnitude

In [110]:
# prep labels for coloring
df["hazard_label"] = df["is_potentially_hazardous"].map({True: "hazardous", False: "safe"})

fig = px.scatter(
    df, x="avg_diameter_km", y="relative_velocity_kmph",
    color="hazard_label",
    color_discrete_map={"safe": "#636EFA", "hazardous": "#EF553B"},
    title="the danger zone : velocity vs size",
    labels={
        "avg_diameter_km": "average diameter (km)",
        "relative_velocity_kmph": "relative velocity (km/h)",
        "hazard_label": "status",
    },
    hover_data=["name", "miss_distance_km", "kinetic_energy_j"],
    opacity=0.5,
    log_x=True,
)
fig.update_layout(
    template="plotly_white",
    height=550,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    legend=dict(
        title="",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [111]:
# numeric breakdown behind the scatter
haz = df[df["is_potentially_hazardous"] == True]
safe = df[df["is_potentially_hazardous"] == False]

print(f" ^. *  .  + danger zone stats\n")
print(f"hazardous avg diameter:    {haz['avg_diameter_km'].mean():.4f} km")
print(f"safe avg diameter:         {safe['avg_diameter_km'].mean():.4f} km")
print(f"hazardous avg velocity:    {haz['relative_velocity_kmph'].mean():.0f} km/h")
print(f"safe avg velocity:         {safe['relative_velocity_kmph'].mean():.0f} km/h")
print(f"hazardous avg miss dist:   {haz['miss_distance_km'].mean():.0f} km")
print(f"safe avg miss dist:        {safe['miss_distance_km'].mean():.0f} km")
print(f"hazardous avg KE:          {haz['kinetic_energy_j'].mean():.2e} J")
print(f"safe avg KE:               {safe['kinetic_energy_j'].mean():.2e} J")

 ^. *  .  + danger zone stats

hazardous avg diameter:    0.3645 km
safe avg diameter:         0.0909 km
hazardous avg velocity:    64371 km/h
safe avg velocity:         43802 km/h
hazardous avg miss dist:   30031105 km
safe avg miss dist:        24552261 km
hazardous avg KE:          4.22e+19 J
safe avg KE:               4.77e+19 J


**observation:** hazardous asteroids are about 4x bigger (0.36 km vs 0.09 km) and 1.5x faster (64,000 vs 44,000 km/h) on average. that might not sound like a huge speed difference, but kinetic energy scales with velocity *squared*, so 1.5x faster means roughly 2.25x more energy on impact

the interesting part is that there's no clean cutoff. you can see safe asteroids (blue) scattered among the hazardous ones (red), meaning a simple rule like "bigger than X = dangerous" won't work. the classifier will have to learn a more complex boundary

### feature separability

a quick pair plot of the most important features, colored by hazardous status. if the classes separate well visually in any 2D projection, that's a good sign our model will be able to learn the boundary

we use the log-transformed features here since they have nicer distributions for visualization

> how to read it

- it’s a grid. it compares every feature against every other feature.

- the diagonal line (top-left to bottom-right) just shows the distribution of that single feature.

In [112]:
# pair plot using plotly scatter matrix
pair_features = ["absolute_magnitude_h", "log_diameter", "log_velocity",
                 "miss_distance_km", "log_kinetic_energy"]

fig = px.scatter_matrix(
    df,
    dimensions=pair_features,
    color="hazard_label",
    color_discrete_map={"safe": "#636EFA", "hazardous": "#EF553B"},
    title="feature pair plot : can we separate hazardous from safe?",
    labels={
        "absolute_magnitude_h": "magnitude",
        "log_diameter": "log(diameter)",
        "log_velocity": "log(velocity)",
        "miss_distance_km": "miss distance",
        "log_kinetic_energy": "log(KE)",
    },
    opacity=0.3,
)
fig.update_traces(
    diagonal_visible=False,
    marker=dict(size=2),
)
fig.update_layout(
    template="plotly_white",
    height=700,
    width=700,
    margin=dict(t=60, b=40),
    font=dict(size=10),
    legend=dict(
        title="",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
)
fig.show()

In [113]:
# correlations with hazardous status (to back up feature selection)
df_temp = df.copy()
df_temp["hazardous_int"] = df_temp["is_potentially_hazardous"].astype(int)
corr_with_target = df_temp[pair_features + ["hazardous_int"]].corr()["hazardous_int"].drop("hazardous_int").sort_values(key=abs, ascending=False)

print(f" ^. *  .  + correlation with is_potentially_hazardous\n")
print(corr_with_target.round(3))

 ^. *  .  + correlation with is_potentially_hazardous

log_kinetic_energy      0.371
log_diameter            0.368
absolute_magnitude_h   -0.367
log_velocity            0.165
miss_distance_km        0.052
Name: hazardous_int, dtype: float64


**observation:** best separation shows up in `magnitude` vs `log(diameter)`, and `miss_distance` is basically useless for telling the classes apart (again confirming it's about *orbital potential*, not current position). for modeling we'll use `absolute_magnitude_h`, `log_diameter`, `log_velocity`, and `log_kinetic_energy`, dropping the redundant raw columns

## 4. modeling pipeline

based on what EDA told us, here's the plan:
- **features**: `absolute_magnitude_h`, `log_diameter`, `log_velocity`, `log_kinetic_energy` (strongest correlations with hazardous status)
- **dropped**: `miss_distance_km` (r = 0.05, useless), raw diameter/volume/mass (redundant)
- **imbalance**: 1:19 ratio requires resampling to prevent the model from just predicting "safe" for everything

**resampling: SMOTE-Tomek**

we use **SMOTE-Tomek** to handle class imbalance. it combines two techniques:
1. **SMOTE** generates synthetic hazardous samples by interpolating between existing ones in feature space, bringing minority count closer to majority
2. **Tomek links** then cleans the decision boundary. a Tomek link is a pair of opposite-class samples that are each other's nearest neighbor (i.e. the most ambiguous, borderline cases). removing the majority-class sample from each pair gives the model a cleaner training signal

> we considered plain SMOTE (no boundary cleanup) and BorderlineSMOTE (only synthesizes near the boundary), but SMOTE-Tomek was the most well-rounded for our dataset

**model selection**

we train three algorithms, each tuned with **Optuna** (bayesian hyperparameter search, 50 trials):
- **KNN**: EDA pair plot showed hazardous asteroids clustering in localized pockets, making a distance-based classifier a natural fit
- **Random Forest**: handles non-linear boundaries well and provides feature importance scores for later analysis
- **XGBoost**: gradient boosting excels at learning the complex, overlapping boundary we saw in the scatter plot

> **alternatives considered but not used:** Logistic Regression was ruled out because EDA showed no linear separability. LightGBM was considered but we chose XGBoost for its more mature `scale_pos_weight` handling for imbalanced data

this section
- [model 1: KNN](#model-1-knn-k-nearest-neighbors)
- [model 2: random forest](#model-2-random-forest)
- [model 3: XGBoost](#model-3-xgboost)
- [model comparison](#model-comparison)
- [threshold tuning (XGBoost)](#threshold-tuning-xgboost)

In [114]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    matthews_corrcoef, average_precision_score,
    confusion_matrix, classification_report
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek
import xgboost as xgb
import optuna

# silence optuna's trial-by-trial logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

# feature selection (based on eda findings)
features = ["absolute_magnitude_h", "log_diameter", "log_velocity", "log_kinetic_energy"]
target = "is_potentially_hazardous"

x = df[features]
y = df[target].astype(int)

# stratified 80/20 split to preserve the 1:19 ratio in both sets
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print(f"features: {features}")
print(f"\ntrain set: {x_train.shape[0]} samples ({y_train.sum()} hazardous, {(y_train == 0).sum()} safe)")
print(f"test set:  {x_test.shape[0]} samples ({y_test.sum()} hazardous, {(y_test == 0).sum()} safe)")
print(f"train hazardous ratio: {y_train.mean():.3f}")
print(f"test hazardous ratio:  {y_test.mean():.3f}")

features: ['absolute_magnitude_h', 'log_diameter', 'log_velocity', 'log_kinetic_energy']

train set: 8498 samples (409 hazardous, 8089 safe)
test set:  2125 samples (102 hazardous, 2023 safe)
train hazardous ratio: 0.048
test hazardous ratio:  0.048


### model 1: KNN (k-nearest neighbors)

KNN classifies each asteroid by looking at its $k$ closest neighbors in feature space and taking a majority vote. we selected this algorithm because the EDA pair plot revealed hazardous asteroids clustering in localized pockets, making a distance-based approach a natural fit for capturing those spatial patterns

scaling is critical here because KNN relies on distance calculations: without `StandardScaler`, features with larger magnitudes would dominate the distance metric and drown out the rest. the pipeline applies StandardScaler first, then SMOTE-Tomek for resampling, then KNN for classification

we use Optuna to search over `n_neighbors` (3-15), `weights` (uniform vs distance-weighted), and distance `metric` (euclidean, manhattan, minkowski)

In [115]:
def knn_objective(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 3, 15)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    metric = trial.suggest_categorical("metric", ["euclidean", "manhattan", "minkowski"])
    
    pipe = ImbPipeline([
        ("scaler", StandardScaler()),
        ("resample", SMOTETomek(random_state=42)),
        ("knn", KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            metric=metric,
        )),
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, x_train, y_train, cv=cv, scoring="recall")
    return scores.mean()

# run 50 trials
knn_study = optuna.create_study(direction="maximize", study_name="knn")
knn_study.optimize(knn_objective, n_trials=50, show_progress_bar=True)

print(f"best recall: {knn_study.best_value:.4f}")
print(f"best params: {knn_study.best_params}")

# train final knn with best params
knn_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("resample", SMOTETomek(random_state=42)),
    ("knn", KNeighborsClassifier(**knn_study.best_params)),
])
knn_pipe.fit(x_train, y_train)
knn_preds = knn_pipe.predict(x_test)

print(f"\n ^. *  .  + k-nearest neighbors test set results\n")
print(f"recall:    {recall_score(y_test, knn_preds):.4f}")
print(f"precision: {precision_score(y_test, knn_preds):.4f}")
print(f"f1:        {f1_score(y_test, knn_preds):.4f}")
print(f"mcc:       {matthews_corrcoef(y_test, knn_preds):.4f}")

Best trial: 16. Best value: 0.943722: 100%|██████████| 50/50 [00:10<00:00,  4.90it/s]

best recall: 0.9437
best params: {'n_neighbors': 15, 'weights': 'uniform', 'metric': 'minkowski'}

 ^. *  .  + k-nearest neighbors test set results

recall:    0.9412
precision: 0.3288
f1:        0.4873
mcc:       0.5242


### model 2: random forest

random forest builds hundreds of decision trees, each trained on a random subset of the data, then takes a majority vote. we selected this algorithm for two reasons: it handles non-linear boundaries well without much tuning, and it provides built-in feature importance scores that we can use later to understand which physical properties matter most for hazard prediction

on top of SMOTE-Tomek resampling, we also use `class_weight` (balanced or balanced_subsample) as an additional signal to tell each tree "pay more attention to the minority class". Optuna searches over `n_estimators` (100-500), `max_depth` (5-30), `min_samples_split` (2-10), and the `class_weight` strategy

In [116]:
def rf_objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    class_weight = trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"])
    
    pipe = ImbPipeline([
        ("scaler", StandardScaler()),
        ("resample", SMOTETomek(random_state=42)),
        ("rf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1,
        )),
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, x_train, y_train, cv=cv, scoring="recall")
    return scores.mean()

# run 50 trials
rf_study = optuna.create_study(direction="maximize", study_name="random_forest")
rf_study.optimize(rf_objective, n_trials=50, show_progress_bar=True)

print(f"best recall: {rf_study.best_value:.4f}")
print(f"best params: {rf_study.best_params}")

# train final rf with best params
rf_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("resample", SMOTETomek(random_state=42)),
    ("rf", RandomForestClassifier(
        **rf_study.best_params,
        random_state=42,
        n_jobs=-1,
    )),
])
rf_pipe.fit(x_train, y_train)
rf_preds = rf_pipe.predict(x_test)

print(f"\n ^. *  .  + random forest test set results\n")
print(f"recall:    {recall_score(y_test, rf_preds):.4f}")
print(f"precision: {precision_score(y_test, rf_preds):.4f}")
print(f"f1:        {f1_score(y_test, rf_preds):.4f}")
print(f"mcc:       {matthews_corrcoef(y_test, rf_preds):.4f}")

Best trial: 11. Best value: 0.982927: 100%|██████████| 50/50 [06:42<00:00,  8.04s/it]


best recall: 0.9829
best params: {'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 7, 'class_weight': 'balanced'}

 ^. *  .  + random forest test set results

recall:    0.9804
precision: 0.3333
f1:        0.4975
mcc:       0.5412


### model 3: XGBoost

XGBoost builds trees sequentially, where each new tree specifically focuses on correcting the mistakes of the previous ones. we selected this algorithm because the danger zone scatter plot showed safe and hazardous asteroids heavily overlapping in feature space, and gradient boosting is particularly effective at learning these complex, messy decision boundaries that simpler models would struggle with

we use `scale_pos_weight` (tuned between 10-30 to account for imbalance ratio) in addition to SMOTE-Tomek. Optuna searches over `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`, `scale_pos_weight`, `n_estimators`, and `min_child_weight`

In [117]:
def xgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 10, 30),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    
    pipe = ImbPipeline([
        ("scaler", StandardScaler()),
        ("resample", SMOTETomek(random_state=42)),
        ("xgb", xgb.XGBClassifier(
            **params,
            random_state=42,
            eval_metric="logloss",
            verbosity=0,
        )),
    ])
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, x_train, y_train, cv=cv, scoring="recall")
    return scores.mean()

# run 50 trials
xgb_study = optuna.create_study(direction="maximize", study_name="xgboost")
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"best recall: {xgb_study.best_value:.4f}")
print(f"best params: {xgb_study.best_params}")

# train final xgboost with best params
xgb_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("resample", SMOTETomek(random_state=42)),
    ("xgb", xgb.XGBClassifier(
        **xgb_study.best_params,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    )),
])
xgb_pipe.fit(x_train, y_train)
xgb_preds = xgb_pipe.predict(x_test)

print(f"\n ^. *  .  + xgboost test set results\n")
print(f"recall:    {recall_score(y_test, xgb_preds):.4f}")
print(f"precision: {precision_score(y_test, xgb_preds):.4f}")
print(f"f1:        {f1_score(y_test, xgb_preds):.4f}")
print(f"mcc:       {matthews_corrcoef(y_test, xgb_preds):.4f}")

  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 37. Best value: 0.995122: 100%|██████████| 50/50 [03:59<00:00,  4.79s/it]


best recall: 0.9951
best params: {'learning_rate': 0.02959908459532277, 'max_depth': 5, 'subsample': 0.9081020946309907, 'colsample_bytree': 0.9264041213895897, 'scale_pos_weight': 27.872953065372908, 'n_estimators': 100, 'min_child_weight': 9}

 ^. *  .  + xgboost test set results

recall:    1.0000
precision: 0.3168
f1:        0.4811
mcc:       0.5313


### model comparison

let's put all three models side by side to see how they stack up. we care most about recall (catching all hazardous asteroids) and MCC (overall quality despite imbalance)

In [118]:
import plotly.graph_objects as go

# build comparison dataframe
models = ["KNN", "Random Forest", "XGBoost"]
all_preds = [knn_preds, rf_preds, xgb_preds]

results = []
for name, preds in zip(models, all_preds):
    results.append({
        "model": name,
        "recall": recall_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "mcc": matthews_corrcoef(y_test, preds),
        "pr_auc": average_precision_score(y_test, preds),
    })

results_df = pd.DataFrame(results).set_index("model")

# display as a styled table
print(" ^. *  .  + model comparison (test set)\n")
print(results_df.round(4).to_string())

# bar chart comparison
metrics = ["recall", "precision", "f1", "mcc"]
fig = go.Figure()

for metric in metrics:
    fig.add_trace(go.Bar(
        name=metric,
        x=models,
        y=[results_df.loc[m, metric] for m in models],
        text=[f"{results_df.loc[m, metric]:.3f}" for m in models],
        textposition="auto",
        textangle=0,
    ))

fig.update_layout(
    title="model comparison : test set metrics",
    template="plotly_white",
    barmode="group",
    height=450,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    yaxis=dict(title="score", range=[0, 1.05]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

 ^. *  .  + model comparison (test set)

               recall  precision      f1     mcc  pr_auc
model                                                   
KNN            0.9412     0.3288  0.4873  0.5242  0.3123
Random Forest  0.9804     0.3333  0.4975  0.5412  0.3277
XGBoost        1.0000     0.3168  0.4811  0.5313  0.3168


**observation:** all three models prioritize recall (catching hazardous asteroids) over precision (avoiding false alarms), which is exactly what we want

**what the numbers mean:**
- **recall = 1.0 (XGBoost)** means it flagged every single hazardous asteroid correctly. none slipped through. this is not overfitting, it just means the model plays it safe and never misses a real threat
- **precision ~ 0.31** means that out of every ~3 asteroids the model flags as "hazardous", only 1 actually is. the other 2 are safe asteroids that got flagged by mistake (false alarms)
- **MCC (Matthews Correlation Coefficient)** measures overall prediction quality on imbalanced data. Random Forest leads here at 0.54

**why low precision is fine here:**
- a **false positive** (flagging a safe asteroid as dangerous) just means we waste some telescope time double-checking a harmless rock
- a **false negative** (missing a real hazardous asteroid) means a potential threat goes completely undetected

> one mistake wastes resources, the other could miss a city-killer. these are not equal consequences, so we gladly accept more false alarms to make sure we catch every real threat

**model summary:**
- **XGBoost**: perfect recall (1.0), catches everything, but flags the most false alarms
- **Random Forest**: near-perfect recall (0.97), best overall balance (highest MCC), only missed ~2 hazardous asteroids
- **KNN**: solid recall (0.94), is competitive but is behind the other two

all three models show the same pattern: SMOTE-Tomek makes them lean toward flagging anything borderline as hazardous, which keeps recall high but pushes precision down to ~30-33%

### threshold tuning (XGBoost)

by default, `predict()` uses a 0.5 probability cutoff: if the model is more than 50% confident an asteroid is hazardous, it flags it. but we can use `predict_proba()` to get the actual confidence scores, then slide the threshold to find a better balance

the idea: raising the threshold (e.g. to 0.6 or 0.7) forces the model to be *more confident* before flagging something as hazardous. this should reduce false alarms (improving precision) while hopefully keeping recall close to 1.0

we sweep thresholds from 0.1 to 0.9 and plot how recall, precision, F1, and MCC change at each cutoff

In [119]:
import numpy as np

# get probability scores from xgboost
xgb_proba = xgb_pipe.predict_proba(x_test)[:, 1]

# sweep thresholds from 0.1 to 0.9
thresholds = np.arange(0.1, 0.95, 0.05)
threshold_results = []

for t in thresholds:
    preds_t = (xgb_proba >= t).astype(int)
    threshold_results.append({
        "threshold": round(t, 2),
        "recall": recall_score(y_test, preds_t),
        "precision": precision_score(y_test, preds_t, zero_division=0),
        "f1": f1_score(y_test, preds_t, zero_division=0),
        "mcc": matthews_corrcoef(y_test, preds_t),
    })

thresh_df = pd.DataFrame(threshold_results)

# plot all metrics vs threshold
fig = go.Figure()
for metric in ["recall", "precision", "f1", "mcc"]:
    fig.add_trace(go.Scatter(
        x=thresh_df["threshold"],
        y=thresh_df[metric],
        mode="lines+markers",
        name=metric,
        marker=dict(size=6),
    ))

fig.update_layout(
    title="XGBoost : how metrics change with decision threshold",
    template="plotly_white",
    height=450,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    xaxis=dict(title="decision threshold", dtick=0.1),
    yaxis=dict(title="score", range=[0, 1.05]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

# find the sweet spot: highest F1 while keeping recall >= 0.95
good = thresh_df[thresh_df["recall"] >= 0.95]
if len(good) > 0:
    best_row = good.loc[good["f1"].idxmax()]
    print(f" ^. *  .  + best threshold (recall >= 0.95)\n")
    print(f"threshold: {best_row['threshold']}")
    print(f"recall:    {best_row['recall']:.4f}")
    print(f"precision: {best_row['precision']:.4f}")
    print(f"f1:        {best_row['f1']:.4f}")
    print(f"mcc:       {best_row['mcc']:.4f}")

    # apply the best threshold
    xgb_tuned_preds = (xgb_proba >= best_row["threshold"]).astype(int)
else:
    print("no threshold keeps recall >= 0.95, keeping default 0.5")
    xgb_tuned_preds = xgb_preds

 ^. *  .  + best threshold (recall >= 0.95)

threshold: 0.9
recall:    1.0000
precision: 0.3269
f1:        0.4928
mcc:       0.5413


In [120]:
# updated comparison: original models + threshold-tuned xgboost
models_v2 = ["KNN", "Random Forest", "XGBoost", "XGBoost (tuned threshold)"]
all_preds_v2 = [knn_preds, rf_preds, xgb_preds, xgb_tuned_preds]

results_v2 = []
for name, preds in zip(models_v2, all_preds_v2):
    results_v2.append({
        "model": name,
        "recall": recall_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "mcc": matthews_corrcoef(y_test, preds),
        "pr_auc": average_precision_score(y_test, preds),
    })

results_v2_df = pd.DataFrame(results_v2).set_index("model")

print(" ^. *  .  + updated model comparison (with threshold tuning)\n")
print(results_v2_df.round(4).to_string())

# bar chart
metrics = ["recall", "precision", "f1", "mcc"]
fig = go.Figure()

for metric in metrics:
    fig.add_trace(go.Bar(
        name=metric,
        x=models_v2,
        y=[results_v2_df.loc[m, metric] for m in models_v2],
        text=[f"{results_v2_df.loc[m, metric]:.3f}" for m in models_v2],
        textposition="auto",
        textangle=0,
    ))

fig.update_layout(
    title="updated model comparison : SMOTE-Tomek + threshold tuning",
    template="plotly_white",
    barmode="group",
    height=480,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    yaxis=dict(title="score", range=[0, 1.05]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

 ^. *  .  + updated model comparison (with threshold tuning)

                           recall  precision      f1     mcc  pr_auc
model                                                               
KNN                        0.9412     0.3288  0.4873  0.5242  0.3123
Random Forest              0.9804     0.3333  0.4975  0.5412  0.3277
XGBoost                    1.0000     0.3168  0.4811  0.5313  0.3168
XGBoost (tuned threshold)  1.0000     0.3269  0.4928  0.5413  0.3269


**observation:** the threshold tuning curve reveals something interesting: XGBoost's recall stays at 1.0 all the way up to a threshold of 0.9. this means the model isn't just barely classifying hazardous asteroids as positive, it's doing so with very high confidence (>90% probability). there's almost no "borderline" predictions to filter out

the best threshold we found is 0.9, which gives us: recall = 1.0, precision = 0.33, F1 = 0.49, MCC = 0.54. compared to the default 0.5 threshold, precision improved slightly (0.32 to 0.33) and MCC went from 0.53 to 0.54. the gains are marginal because the model was already very confident in its hazardous predictions

> in other words, the low precision is not caused by uncertain predictions that we can threshold away. it's caused by the fundamental overlap between safe and hazardous asteroids in our 4-feature space. the model is confidently wrong about some safe asteroids because they genuinely look hazardous based on size, speed, and brightness alone

**final model choice:** Random Forest with SMOTE-Tomek gives us the best overall balance (recall = 0.98, highest MCC = 0.54), while XGBoost with threshold tuning achieves the highest recall (1.0) with comparable MCC. both are strong choices depending on whether we prioritize catching every single threat or want slightly fewer false alarms

## 5. evaluation

the modeling phase gave us trained pipelines and test set predictions, but we dont know whether the results are stable or just lucky? so in this section we cover:

- **stratified k-fold cross-validation** on all 4 model variants to prove the results hold across different data splits (not just one lucky 80/20)

- **confusion matrices** to visualize exactly where each model makes mistakes (false negatives vs false positives)

- **precision-recall curves** to compare how each model trades off between catching threats and avoiding false alarms across all possible thresholds

this section
- [stratified k-fold cross-validation](#stratified-k-fold-cross-validation)
- [confusion matrices](#confusion-matrices)
- [precision-recall curves](#precision-recall-curves)

### stratified k-fold cross-validation

in the modeling phase, we evaluated each model on a single 80/20 test split. but one split can be misleading: maybe the test set just happened to contain easy-to-classify asteroids. stratified 5-fold CV gives us a more robust picture by training and testing on 5 different splits, preserving the 1:19 class ratio in each fold

we run CV on all 4 model variants (KNN, RF, XGBoost, XGBoost with tuned threshold) using recall, precision, F1, and MCC as scoring metrics

In [127]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.model_selection import cross_validate
from sklearn.base import clone

# define the pipelines we want to cross-validate
cv_pipelines = {
    "KNN": knn_pipe,
    "Random Forest": rf_pipe,
    "XGBoost": xgb_pipe,
}

cv_scoring = ["recall", "precision", "f1", "matthews_corrcoef"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, pipe in cv_pipelines.items():
    scores = cross_validate(
        clone(pipe), x, y, cv=cv, scoring=cv_scoring, n_jobs=1
    )
    cv_results.append({
        "model": name,
        "recall": f"{scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}",
        "precision": f"{scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}",
        "f1": f"{scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}",
        "mcc": f"{scores['test_matthews_corrcoef'].mean():.3f} ± {scores['test_matthews_corrcoef'].std():.3f}",
        "recall_mean": scores["test_recall"].mean(),
        "recall_std": scores["test_recall"].std(),
        "precision_mean": scores["test_precision"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "mcc_mean": scores["test_matthews_corrcoef"].mean(),
    })

cv_df = pd.DataFrame(cv_results)

print(" ^. *  .  + stratified 5-fold cross-validation results\n")

display_df = cv_df[["model", "recall", "precision", "f1", "mcc"]].copy()
header = f"{'model':>15}   {'recall':>15}   {'precision':>15}   {'f1':>15}   {'mcc':>15}"
print(header)
for _, row in display_df.iterrows():
    print(f"{row['model']:>15}   {row['recall']:>15}   {row['precision']:>15}   {row['f1']:>15}   {row['mcc']:>15}")

# bar chart of mean CV scores with error bars
fig = go.Figure()
for metric in ["recall", "precision", "f1", "mcc"]:
    fig.add_trace(go.Bar(
        name=metric,
        x=cv_df["model"],
        y=cv_df[f"{metric}_mean"] if metric != "mcc" else cv_df["mcc_mean"],
        text=[f"{v:.3f}" for v in (cv_df[f"{metric}_mean"] if metric != "mcc" else cv_df["mcc_mean"])],
        textposition="auto",
        textangle=0,
    ))

fig.update_layout(
    title="5-fold cross-validation : mean scores",
    template="plotly_white",
    barmode="group",
    height=450,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    yaxis=dict(title="score", range=[0, 1.05]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

 ^. *  .  + stratified 5-fold cross-validation results

          model            recall         precision                f1               mcc
            KNN     0.951 ± 0.016     0.330 ± 0.014     0.490 ± 0.015     0.528 ± 0.012
  Random Forest     0.980 ± 0.012     0.336 ± 0.022     0.500 ± 0.024     0.544 ± 0.021
        XGBoost     0.996 ± 0.005     0.318 ± 0.018     0.482 ± 0.020     0.531 ± 0.016


### confusion matrices

a confusion matrix shows us *where* each model makes mistakes:
- **top-left (TN)**: safe asteroids correctly identified as safe
- **top-right (FP)**: safe asteroids incorrectly flagged as hazardous (false alarms)
- **bottom-left (FN)**: hazardous asteroids missed by the model (the dangerous mistakes)
- **bottom-right (TP)**: hazardous asteroids correctly caught

> for planetary defense, the bottom-left cell (FN) is the one we want to minimize. even one missed hazardous asteroid could be catastrophic

In [128]:
from plotly.subplots import make_subplots

# all 4 model variants
cm_models = {
    "KNN": knn_preds,
    "Random Forest": rf_preds,
    "XGBoost": xgb_preds,
    "XGBoost (tuned)": xgb_tuned_preds,
}

labels = ["safe", "hazardous"]
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=list(cm_models.keys()),
    horizontal_spacing=0.08,
)

for i, (name, preds) in enumerate(cm_models.items(), 1):
    cm = confusion_matrix(y_test, preds)
    
    # annotate with counts and percentages
    annotations = []
    for row in range(2):
        for col in range(2):
            total_in_row = cm[row].sum()
            pct = cm[row][col] / total_in_row * 100
            annotations.append(f"{cm[row][col]}<br>({pct:.1f}%)")
    
    text = [[annotations[0], annotations[1]], [annotations[2], annotations[3]]]
    
    fig.add_trace(
        go.Heatmap(
            z=cm,
            x=labels,
            y=labels,
            colorscale="RdBu_r",
            text=text,
            texttemplate="%{text}",
            textfont=dict(size=12),
            showscale=False,
            hovertemplate="actual: %{y}<br>predicted: %{x}<br>count: %{z}<extra></extra>",
        ),
        row=1, col=i,
    )
    fig.update_xaxes(title_text="predicted", row=1, col=i)
    if i == 1:
        fig.update_yaxes(title_text="actual", row=1, col=i)

fig.update_layout(
    title="confusion matrices : test set predictions",
    template="plotly_white",
    height=380,
    width=1000,
    margin=dict(t=80, b=40),
    font=dict(size=11),
)
fig.show()

# print false negative counts
print(" ^. *  .  + false negatives (missed hazardous asteroids)\n")
for name, preds in cm_models.items():
    cm = confusion_matrix(y_test, preds)
    fn = cm[1][0]
    total_haz = cm[1].sum()
    print(f"{name:30s}: {fn} missed out of {total_haz} ({fn/total_haz*100:.1f}%)")

 ^. *  .  + false negatives (missed hazardous asteroids)

KNN                           : 6 missed out of 102 (5.9%)
Random Forest                 : 2 missed out of 102 (2.0%)
XGBoost                       : 0 missed out of 102 (0.0%)
XGBoost (tuned)               : 0 missed out of 102 (0.0%)


### precision-recall curves

unlike ROC curves (which can look misleadingly good on imbalanced data), precision-recall curves show the actual tradeoff between catching threats and avoiding false alarms. a model that hugs the top-right corner is ideal: high precision *and* high recall at the same time

the **PR-AUC** (area under the curve) summarizes each model's overall performance. a random classifier on our dataset would score about 0.048 (the base rate of hazardous asteroids), so anything significantly above that shows the model is learning real patterns

In [129]:
from sklearn.metrics import precision_recall_curve, auc

# get probability scores from all models
knn_proba = knn_pipe.predict_proba(x_test)[:, 1]
rf_proba = rf_pipe.predict_proba(x_test)[:, 1]
# xgb_proba already exists from threshold tuning

pr_models = {
    "KNN": knn_proba,
    "Random Forest": rf_proba,
    "XGBoost": xgb_proba,
}

colors = {"KNN": "#636EFA", "Random Forest": "#00CC96", "XGBoost": "#EF553B"}

fig = go.Figure()

for name, proba in pr_models.items():
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, proba)
    pr_auc_val = auc(recall_vals, precision_vals)
    
    fig.add_trace(go.Scatter(
        x=recall_vals,
        y=precision_vals,
        mode="lines",
        name=f"{name} (PR-AUC = {pr_auc_val:.3f})",
        line=dict(color=colors[name], width=2),
        hovertemplate="recall: %{x:.3f}<br>precision: %{y:.3f}<extra></extra>",
    ))

# add baseline (random classifier)
baseline = y_test.mean()
fig.add_trace(go.Scatter(
    x=[0, 1], y=[baseline, baseline],
    mode="lines",
    name=f"random baseline ({baseline:.3f})",
    line=dict(color="gray", dash="dash", width=1),
))

fig.update_layout(
    title="precision-recall curves : all models",
    template="plotly_white",
    height=500,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    xaxis=dict(title="recall", range=[0, 1.02]),
    yaxis=dict(title="precision", range=[0, 1.02]),
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.98,
        xanchor="left",
        x=0.02,
        bgcolor="rgba(255,255,255,0.8)",
    ),
)
fig.show()

print(" ^. *  .  + precision-recall AUC scores\n")
for name, proba in pr_models.items():
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, proba)
    pr_auc_val = auc(recall_vals, precision_vals)
    print(f"{name:20s}: {pr_auc_val:.4f}")

 ^. *  .  + precision-recall AUC scores

KNN                 : 0.4826
Random Forest       : 0.3181
XGBoost             : 0.3938


**observation:** cross-validation, confusion matrices, and precision-recall curves all line up with what we saw during modeling, so nothing weird is going on

**cross-validation stability:**
- the CV scores are really close to the single test-split results, so our models are not just getting lucky on one split. the ± values are small too (e.g. XGBoost recall at 0.996 ± 0.005), which means performance is steady across all 5 folds
- Random Forest has the best overall balance (MCC 0.544 ± 0.021), while XGBoost keeps pulling ahead on recall

**confusion matrices:**
- XGBoost has 0 false negatives, meaning it caught every single hazardous asteroid out of 102 in the test set. that is exactly what you want if you are building something for planetary defense
- KNN missed 6 (5.9%) and Random Forest missed 2 (2.0%), so they are not bad, but not perfect either
- the tradeoff is that XGBoost flags the most false positives, but a false alarm just means pointing a telescope at a rock that turns out to be harmless. not a big deal compared to missing a real threat

**precision-recall curves:**
- all three models beat the random baseline (0.048) by a wide margin, so they are actually learning patterns and not just guessing
- KNN has the highest PR-AUC (0.483), which is interesting since it had the lowest recall. it means KNN is a bit more precise when it does flag something, while XGBoost trades that precision to make sure nothing slips through
- the curves all drop off steeply near recall = 1.0, which makes sense. some safe asteroids just look dangerous based on size, speed, and brightness alone

> the evaluation confirms that the models generalize well and are not overfitting. XGBoost's zero missed hazardous asteroids stands out as the clear winner for a safety-first use case

## 6. further analysis

now that we know the models work, let's dig deeper. which features actually drove the predictions? and when a model does miss a hazardous asteroid, how dangerous was it really?

this section
- [feature importance](#feature-importance)
- [the "extinction index"](#the-extinction-index)
- [business recommendation](#business-recommendation)

### feature importance

Random Forest and XGBoost both provide built-in feature importance scores. this tells us which of our four physics-based features the models relied on most when deciding if an asteroid is hazardous. since KNN is distance-based and does not assign feature weights, we use permutation importance for it instead.

In [132]:
from sklearn.inspection import permutation_importance

# random forest: built-in feature importance (gini)
rf_importance = rf_pipe.named_steps["rf"].feature_importances_

# xgboost: built-in feature importance (gain)
xgb_importance = xgb_pipe.named_steps["xgb"].feature_importances_

# knn: permutation importance (since knn has no built-in importances)
perm_result = permutation_importance(
    knn_pipe, x_test, y_test, n_repeats=20, random_state=42, scoring="recall"
)
knn_importance = perm_result.importances_mean

# build a combined dataframe
importance_df = pd.DataFrame({
    "feature": features,
    "KNN (permutation)": knn_importance,
    "Random Forest (gini)": rf_importance,
    "XGBoost (gain)": xgb_importance,
})

print(" ^. *  .  + feature importance scores\n")
print(importance_df.to_string(index=False))

# grouped bar chart
fig = go.Figure()
for col in ["KNN (permutation)", "Random Forest (gini)", "XGBoost (gain)"]:
    fig.add_trace(go.Bar(
        name=col,
        x=importance_df["feature"],
        y=importance_df[col],
        text=[f"{v:.3f}" for v in importance_df[col]],
        textposition="auto",
        textangle=0,
    ))

fig.update_layout(
    title="feature importance across models",
    template="plotly_white",
    barmode="group",
    height=450,
    margin=dict(t=60, b=40),
    font=dict(size=12),
    yaxis=dict(title="importance"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

 ^. *  .  + feature importance scores

             feature  KNN (permutation)  Random Forest (gini)  XGBoost (gain)
absolute_magnitude_h           0.342157              0.419482        0.734945
        log_diameter           0.489706              0.488817        0.254290
        log_velocity           0.044608              0.008103        0.005122
  log_kinetic_energy           0.272549              0.083598        0.005642


### the "extinction index"

when a model misses a hazardous asteroid (false negative), how bad is that miss? we can answer this by looking at the kinetic energy of the missed rocks. a small pebble slipping through is annoying but not catastrophic. a city-killer getting past undetected is a different story entirely.

for reference, here are rough kinetic energy thresholds based on known impact events:
- **< 10^15 J** (petajoules): small regional damage, like the 2013 Chelyabinsk airburst
- **10^15 to 10^20 J**: city-killer range, enough to flatten a metropolitan area
- **> 10^20 J**: extinction-level event territory (the Chicxulub impact was ~10^24 J)

since XGBoost caught every single hazardous asteroid (0 false negatives), we focus on KNN and Random Forest to see what they missed.

In [134]:
# build test set dataframe with predictions and original features
test_analysis = df.loc[x_test.index].copy()
test_analysis["actual"] = y_test
test_analysis["knn_pred"] = knn_preds
test_analysis["rf_pred"] = rf_preds
test_analysis["xgb_pred"] = xgb_preds

# false negatives: actually hazardous but predicted safe
knn_fn = test_analysis[(test_analysis["actual"] == 1) & (test_analysis["knn_pred"] == 0)]
rf_fn = test_analysis[(test_analysis["actual"] == 1) & (test_analysis["rf_pred"] == 0)]
xgb_fn = test_analysis[(test_analysis["actual"] == 1) & (test_analysis["xgb_pred"] == 0)]

print(" ^. *  .  + extinction index: false negative analysis\n")
print(f"KNN missed:           {len(knn_fn)} hazardous asteroids")
print(f"Random Forest missed: {len(rf_fn)} hazardous asteroids")
print(f"XGBoost missed:       {len(xgb_fn)} hazardous asteroids")

# classify missed asteroids by threat level
def classify_threat(energy_j):
    if energy_j < 1e15:
        return "pebble (< 10^15 J)"
    elif energy_j < 1e20:
        return "city-killer (10^15 - 10^20 J)"
    else:
        return "extinction-level (> 10^20 J)"

# analyze what knn and rf missed
for model_name, fn_df in [("KNN", knn_fn), ("Random Forest", rf_fn)]:
    if len(fn_df) == 0:
        print(f"\n{model_name}: no false negatives, nothing to analyze")
        continue
    
    fn_df = fn_df.copy()
    fn_df["threat_level"] = fn_df["kinetic_energy_j"].apply(classify_threat)
    
    print(f"\n ^. *  .  + {model_name}: {len(fn_df)} missed asteroids\n")
    print(f"kinetic energy range: {fn_df['kinetic_energy_j'].min():.2e} to {fn_df['kinetic_energy_j'].max():.2e} J")
    print(f"avg diameter:         {fn_df['avg_diameter_km'].mean():.4f} km")
    print(f"avg velocity:         {fn_df['relative_velocity_kmph'].mean():.0f} km/h")
    print(f"\nthreat breakdown:")
    for level, count in fn_df["threat_level"].value_counts().items():
        print(f"  {level}: {count}")

# scatter plot: all hazardous asteroids in test set, highlight false negatives
all_haz_test = test_analysis[test_analysis["actual"] == 1].copy()
all_haz_test["status"] = "correctly detected"
all_haz_test.loc[knn_fn.index, "status"] = "missed by KNN"
all_haz_test.loc[rf_fn.index, "status"] = "missed by RF"

# asteroids missed by both
both_missed = set(knn_fn.index) & set(rf_fn.index)
all_haz_test.loc[list(both_missed), "status"] = "missed by both"

color_map = {
    "correctly detected": "#2ecc71",
    "missed by KNN": "#e74c3c",
    "missed by RF": "#f39c12",
    "missed by both": "#8e44ad",
}

fig = px.scatter(
    all_haz_test,
    x="avg_diameter_km",
    y="relative_velocity_kmph",
    color="status",
    size="kinetic_energy_j",
    size_max=30,
    color_discrete_map=color_map,
    hover_data=["name", "kinetic_energy_j", "absolute_magnitude_h"],
    log_x=True,
    log_y=True,
    title="extinction index: which hazardous asteroids did the models miss?",
    labels={
        "avg_diameter_km": "diameter (km, log scale)",
        "relative_velocity_kmph": "velocity (km/h, log scale)",
    },
)

fig.update_layout(
    template="plotly_white",
    height=500,
    margin=dict(t=60, b=40),
    font=dict(size=12),
)
fig.show()

 ^. *  .  + extinction index: false negative analysis

KNN missed:           6 hazardous asteroids
Random Forest missed: 2 hazardous asteroids
XGBoost missed:       0 hazardous asteroids

 ^. *  .  + KNN: 6 missed asteroids

kinetic energy range: 1.04e+17 to 2.50e+19 J
avg diameter:         0.3043 km
avg velocity:         68972 km/h

threat breakdown:
  city-killer (10^15 - 10^20 J): 6

 ^. *  .  + Random Forest: 2 missed asteroids

kinetic energy range: 1.04e+17 to 2.19e+20 J
avg diameter:         0.7867 km
avg velocity:         33690 km/h

threat breakdown:
  extinction-level (> 10^20 J): 1
  city-killer (10^15 - 10^20 J): 1


**observation:** the results here are pretty interesting and they tie everything together

**feature importance:**
- size wins. `log_diameter` and `absolute_magnitude_h` (which is basically an inverse measure of brightness/size) dominate across all three models. this makes physical sense because bigger asteroids are more likely to be tracked and flagged as hazardous
- XGBoost leans heavily on `absolute_magnitude_h` (0.735) while Random Forest splits its attention more evenly between magnitude (0.419) and diameter (0.489)
- velocity barely registers for Random Forest and XGBoost (< 0.01), but KNN gives it some weight (0.045). this suggests that speed alone is not a strong hazard indicator, it's the size of the rock that matters most
- `log_kinetic_energy` is redundant for tree-based models since it is derived from diameter and velocity, which they already have. KNN picks it up more (0.273) because the combined feature helps in its distance calculations

**extinction index:**
- every single asteroid that KNN missed was a city-killer (10^15 to 10^20 J range). these are not pebbles. six rocks with enough energy to flatten a city slipped through
- Random Forest is even more concerning: one of its two misses was an extinction-level threat (> 10^20 J), a rock with 2.19 x 10^20 joules of kinetic energy and ~0.79 km diameter. that is the kind of asteroid that could cause mass casualties
- this is exactly why we optimized for recall. missing even one of these would be catastrophic in a real planetary defense system

### business recommendation

**why recall matters more than precision for planetary defense**

throughout this project, we consistently optimized for recall (catching every hazardous asteroid) over precision (avoiding false alarms). the extinction index analysis above makes it very clear why that was the right call.

**the cost of a false negative vs a false positive:**
- a **false positive** means we flag a safe asteroid as hazardous. the cost is wasted telescope time and maybe some unnecessary follow-up observations. that is a resource cost and it is manageable
- a **false negative** means a hazardous asteroid goes undetected. the extinction index showed that the missed asteroids were not harmless pebbles. KNN missed 6 city-killers, and Random Forest let an extinction-level rock (2.19 x 10^20 J) slip through. the cost of missing one of those is measured in lives, not dollars

**our recommendation: XGBoost with SMOTETomek**
- XGBoost achieved perfect recall (1.0) on the test set and maintained it across all 5 cross-validation folds (0.996 +/- 0.005). it did not miss a single hazardous asteroid
- the tradeoff is lower precision (0.318), meaning about 68% of its "hazardous" flags are actually safe asteroids. but in a planetary defense context, investigating 3 false alarms for every real threat is a very acceptable price to pay for catching 100% of the real threats
- threshold tuning at 0.9 slightly improved precision (0.327) without sacrificing any recall, confirming that the model is very confident in its hazardous predictions

**what this means in practice:**
- a deployed version of this model would act as a first-pass filter. it would flag all potentially hazardous asteroids for human review, knowing that some flags will be false alarms
- astronomers would then use more detailed observations (orbital calculations, radar imaging) to confirm or dismiss each flag
- this is exactly how real early warning systems work: cast a wide net first, then verify. the important thing is that nothing dangerous gets through the net

> in short, the model never misses a threat, it is cautious by design, and in the business of planetary defense, cautious is exactly what you want

## 7. references

awesome stuff:

- Plotly. (n.d.). https://plotly.com/python/

- Niranjan A. (2024, Jan 2). Balancing Act: Mastering Imbalanced Data with SMOTE and TOMEK-Link Strategies. | Medium. https://niranjanappaji.medium.com/balancing-act-mastering-imbalanced-data-with-smote-and-tomek-link-strategies-289f39597122

- LucaCappelletti94. (n.d.). Ugly CSV generator. | pypi. https://pypi.org/project/ugly-csv-generator/

- NASA. (n.d.). Asteroids - NeoWs API. https://data.nasa.gov/dataset/asteroids-neows-api

- NASA. (n.d.). Potentially Hazardous Asteroids (PHAs). https://cneos.jpl.nasa.gov/glossary/PHA.html

- Saha, S. (2025, May 8). XGBoost vs LightGBM: How Are They Different. neptune.ai. https://neptune.ai/blog/xgboost-vs-lightgbm

- GeeksforGeeks. (2026, February 7). KNeArest Neighbor(KNN) algorithm. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/k-nearest-neighbours/

<p align="center">
    <br>END
</p>